# Application and reliability checks
Verify the local Streamlit server and run the project's automated checks, including real study upload, prediction rendering, Grad-CAM, exports, invalid-input messages, stale-result prevention and clearing the case.

To launch the app, double-click **Launch KneeAssist AI.cmd** in the project folder. No training notebook is needed for inference.


In [1]:
from pathlib import Path
import sys, json
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "config.yaml").exists() and (p / "src").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from KneeAssist-AI or its notebooks folder.")
sys.path.insert(0, str(ROOT))
from src.utils import config
cfg = config(ROOT / "config.yaml")
print("Project:", ROOT)
print("Python:", sys.executable)


Project: C:\Users\HP\Desktop\KneeAssist-AI
Python: C:\Users\HP\Desktop\KneeAssist-AI\.venv\Scripts\python.exe


In [2]:
import urllib.request
from IPython.display import display, Markdown
url = f"http://{cfg['ui']['host']}:{cfg['ui']['port']}"
try:
    with urllib.request.urlopen(url + "/_stcore/health", timeout=5) as response:
        assert response.status == 200
    print("Streamlit health check passed.")
except Exception as error:
    print("Start Launch KneeAssist AI.cmd to make the dashboard available:", error)
display(Markdown(f"[Open KneeAssist AI dashboard]({url})"))


Streamlit health check passed.


[Open KneeAssist AI dashboard](http://127.0.0.1:8501)

In [3]:
import subprocess
completed = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q"], cwd=ROOT, capture_output=True, text=True)
print(completed.stdout)
if completed.returncode:
    print(completed.stderr[-10000:])
    raise RuntimeError("One or more reliability checks failed. Do not treat the application as verified.")


.......................                                                  [100%]
23 passed in 9.44s



In [4]:
print('Historical diagnostics from the original run; they do not describe a new resume experiment.')
for filename in ["results/leakage_audit.json", "results/diagnostics/resume_verified.json", "results/diagnostics/gpu_precision_findings.json"]:
    print(filename)
    display(json.loads((ROOT / filename).read_text()))
print("Selected model checkpoint exists:", (ROOT / cfg['evaluation']['checkpoint']).is_file())


Historical diagnostics from the original run; they do not describe a new resume experiment.
results/leakage_audit.json


{'dataset': 'MRNet-v1.0',
 'split_counts': {'fit': 904, 'tune': 226, 'official_valid': 120},
 'study_overlap': 0,
 'exact_series_overlap_across_splits': 0,
 'within_partition_duplicate_series': 0,
 'patient_grouping_verified': False,
 'limitation': 'Patient linkage is absent from this MRNet copy; study separation cannot prove patient independence.',
 'prior_evaluation_disclosure': 'The official 120-exam validation set was previously evaluated in the earlier baseline; it is not a new external test set.'}

results/diagnostics/resume_verified.json


{'interrupted_after_epoch': 2,
 'resumed_at_epoch': 3,
 'reached_epoch': 15,
 'passed': True,
 'preserved': ['model',
  'optimizer',
  'scheduler',
  'scaler',
  'random_states',
  'history']}

results/diagnostics/gpu_precision_findings.json


{'device': 'NVIDIA T500 (4 GB)',
 'pytorch': '2.11.0+cu128',
 'encoder_fp32_finite': True,
 'encoder_fp16_finite': False,
 'classifier_amp_finite': True,
 'resolution': 'Frozen encoder uses FP32. Trainable head uses CUDA autocast and GradScaler.',
 'trial_input': 'MRNet training study 0000 axial sequence',
 'first_nonfinite_resnet_sequential_layer': 5}

Selected model checkpoint exists: True


## Release acceptance
The final release should also have a browser-level check with the real validation ZIP, visible probabilities and an attention overlay. The final evidence is written to `results/verification.json`. Automated tests and working software do not establish clinical validity or regulatory clearance.
